# Адаптивная регуляризация в методе Гаусса–Ньютона

**Цель.** В прежней схеме (`run_optimization` / `compute_delta_gn`, удалены из
библиотеки; здесь воспроизведены как `run_optimization_manual`) три параметра регуляризации
задаются руками и подбираются под каждую задачу: $\mu$ (с расписанием
`mu_dec`), $\lambda$ (демпфер Левенберга–Марквардта) и $\lambda_{\mathrm{reg}}$
(гребень). Здесь разбирается, что эти параметры означают на самом деле, и
строится схема, в которой они подбираются автоматически по ходу оптимизации.

**Содержание.**
1. Текущая схема и её слабые места.
2. Что на самом деле делает $\mu$ (исключение множителей $\Rightarrow$ штраф $1/\mu$).
3. Адаптивный демпфер $\lambda$: gain ratio (Нильсен).
4. Адаптивное $\mu$: автоматический старт по кривизне + ужесточение по Пауэллу
   с гейтом по измерениям.
5. Согласованный критерий принятия шага и остановка.
6. Алгоритм целиком и разбор реализации (`gauss_newton/adaptive.py`).
7. Эксперименты (Лотка–Вольтерра, Аттрактор Лоренца).
8. Реальные данные (CeedEveron).
9. Augmented Lagrangian: множители вместо $\mu\to 0$ — вывод формул и
   почему на этой задаче они не выигрывают.
10. Выводы; тесты — `pytests/adaptive_test.py`.


## 1. Текущая схема и её слабые места

Неизвестные $p = [\theta;\, c_1 \dots c_T]$ (параметры + начальные состояния
шутов). Взвешенные измерительные невязки $R(p)$ с якобианом $J$
(в коде $R = W\,(y - h(x))$, $J = W\,\partial h/\partial p$, поэтому
$R(p+\delta) \approx R - J\delta$) и невязки стыковки $R_G(p)$ с якобианом
$J_G$ (знаки согласованы так, что линеаризация невязки стыковки —
$J_G\delta - R_G$).

Шаг прежней схемы — решение седловой системы (сегодня её решает `gn_step`):

$$
\begin{bmatrix}
H + \lambda_{\mathrm{reg}} I + \lambda\operatorname{diag}(H) & J_G^\top\\[2pt]
J_G & -\mu I
\end{bmatrix}
\begin{bmatrix}\delta\\ \nu\end{bmatrix}
=
\begin{bmatrix}J^\top R\\ R_G\end{bmatrix},
\qquad H = J^\top J .
$$

Управление в прежней схеме: после успешного шага $\mu \leftarrow
\mu\cdot\texttt{mu\_dec}$ (не ниже $\mu_{\min}$), после неудачного $\mu
\leftarrow \mu/\texttt{mu\_dec}$; шаг принимается, если
$\|R\|^2+\|R_G\|^2 \le 1.1\cdot\text{best}$.

**Слабые места.**

1. $\mu_0$, `mu_dec` — размерные величины, подбираются руками под задачу
   (в тестах: ЛВ $\mu=0$, Аттрактор $\mu=100$; в ноутбуке с реальными данными
   $\mu=0.03$). Ниже показано: на Аттракторе $\mu_0=0.01$ даёт расходимость
   с NaN, а $\mu_0=1{-}10^4$ — сходимость. Цена ошибки несимметрична и заранее
   неизвестна.
2. Критерий «$\le 1.1\cdot$best» допускает рост стоимости до 10% на каждом
   шаге — возможен монотонный дрейф вверх без остановки.
3. Несогласованность модели и критерия: шаг минимизирует одну функцию
   (см. §2 — это Гаусс–Ньютон для $\Phi_\mu$), а принимается по другой
   ($\|R\|^2+\|R_G\|^2$). Хорошие для $\Phi_\mu$ шаги могут отклоняться,
   плохие — приниматься.
4. $\lambda$ и $\lambda_{\mathrm{reg}}$ фиксированы навсегда, хотя именно
   демпфер должен управлять длиной шага при неудачах; вместо него при
   неудаче трогают $\mu$, меняя тем самым саму целевую функцию. В single
   shooting ($J_G$ пусто) $\lambda$ вообще игнорируется (жёсткий гребень
   $10^{-8}$).
5. Нет критерия остановки — всегда сжигаются все `n_iter` итераций
   (на реальных данных 30 solve по $\sim$0.4 с, даже если сошлись за 6).


## 2. Что на самом деле делает $\mu$

Исключим множители из седловой системы. Из нижней строки
$\nu = (J_G\delta - R_G)/\mu$; подстановка в верхнюю даёт

$$
\Big(H + \lambda_{\mathrm{reg}} I + \lambda\operatorname{diag}(H)
+ \tfrac1\mu J_G^\top J_G\Big)\,\delta
= J^\top R + \tfrac1\mu J_G^\top R_G .
$$

Это в точности нормальные уравнения демпфированного Гаусса–Ньютона для
взвешенной задачи

$$
\boxed{\;\Phi_\mu(p) = \|R(p)\|^2 + \tfrac1\mu\,\|R_G(p)\|^2\;}
$$

То есть $\mu$ — **не** «малая регуляризация», а **обратный вес невязок
стыковки**:

- $\mu$ велико — стыковка мягкая, шуты почти независимы: каждый сегмент
  подгоняется к своим измерениям, задача хорошо глобализуется;
- $\mu \to 0$ — жёсткая стыковка: переход к (почти) точному ограничению
  непрерывности.

Расписание `mu_dec` — это *penalty continuation* (гомотопия по жёсткости
ограничений). Побочная роль блока $-\mu I$ — регуляризация седловой системы
(проксимальный член по двойственным переменным): при $\mu = 0$ и вырожденном
$J_G$ система необратима.

Связь с методом модифицированных функций Лагранжа: если добавить в правую
часть оценку множителей $\hat\nu$, получится augmented Lagrangian; текущая
схема — частный случай $\hat\nu = 0$ (метод квадратичного штрафа в седловой
форме, он же стабилизированный SQP с нулевыми множителями). Полноценный AL —
вывод формул и эксперименты в §9: на этой задаче с ГН-приближением гессиана
он не выигрывает, феасибилити надёжнее достигается continuation по $\mu$.

**Следствия для адаптивности.**
(а) Принимать шаг надо по $\Phi_\mu$ с тем же $\mu$, что использовался при
вычислении шага — иначе модель и критерий говорят о разных функциях.
(б) У $\mu$ есть ясный смысл, значит его можно привязать к наблюдаемым
величинам задачи (кривизна, невязка стыковки), а не к ручному расписанию.


## 3. Адаптивный демпфер $\lambda$: gain ratio (Нильсен)

Квадратичная модель $\Phi_\mu$ в текущей точке:

$$
m(\delta) = \|R - J\delta\|^2 + \tfrac1\mu\|J_G\delta - R_G\|^2 ,
$$

шаг — решение $(\tilde H + D)\,\delta = \tilde g$, где
$\tilde H = H + \tfrac1\mu J_G^\top J_G$,
$\tilde g = J^\top R + \tfrac1\mu J_G^\top R_G$,
$D = \lambda_{\mathrm{reg}} I + \lambda \operatorname{diag}(H)$ — демпфер
(решаем в седловой форме, как в §1, — она лучше обусловлена при малых $\mu$).

**Предсказанное моделью уменьшение — полный вывод.** Раскроем квадраты в
$m(\delta)$; линейные члены собираются в $\tilde g$, квадратичные — в
$\tilde H$:

$$
m(\delta) = m(0) - 2\,\delta^\top\tilde g + \delta^\top\tilde H\,\delta,
\qquad m(0) = \Phi_\mu(p),
$$

откуда

$$
\mathrm{pred} \;=\; m(0) - m(\delta) \;=\; 2\,\delta^\top\tilde g - \delta^\top\tilde H\,\delta .
$$

Шаг удовлетворяет $(\tilde H + D)\delta = \tilde g$, то есть
$\tilde H\delta = \tilde g - D\delta$. Подставляя:

$$
\mathrm{pred} = 2\,\delta^\top\tilde g - \delta^\top(\tilde g - D\delta)
= \boxed{\;\delta^\top\tilde g + \delta^\top D\,\delta\;}
$$

Оба слагаемых неотрицательны: $\delta^\top\tilde g =
\delta^\top(\tilde H + D)\delta \ge 0$ (обе матрицы положительно
полуопределены) и $\delta^\top D\,\delta \ge 0$. Поэтому
$\mathrm{pred} > 0$ всегда, пока система решена корректно —
$\mathrm{pred} \le 0$ служит дешёвым детектором численного сбоя
факторизации.

**Gain ratio:**

$$
\rho = \frac{\Phi_\mu(p) - \Phi_\mu(p+\delta)}{\mathrm{pred}}
$$

— отношение фактического уменьшения к предсказанному: $\rho \approx 1$ —
линеаризация хороша (можно доверять большим шагам), $\rho \le 0$ — шаг
вредный (фактическая стоимость выросла).

**Обновление $\lambda$** (Nielsen, 1999; де-факто стандарт — Ceres,
наследники MINPACK):

- $\rho > 0$ (принять): $\lambda \leftarrow \lambda\cdot\max\!\big(\tfrac13,\,
  1 - (2\rho-1)^3\big)$, $\;\nu_{\mathrm{esc}} \leftarrow 2$;
- $\rho \le 0$ (отклонить): $\lambda \leftarrow \lambda\cdot\nu_{\mathrm{esc}}$,
  $\;\nu_{\mathrm{esc}} \leftarrow 2\nu_{\mathrm{esc}}$ (экспоненциальная
  эскалация).

Форма множителя $\max(1/3,\,1-(2\rho-1)^3)$: при $\rho=1$ (модель идеальна)
$\lambda$ делится на 3; при $\rho=0.5$ — не меняется; при $\rho\to 0^+$ —
почти не растёт (шаг принят, но доверие не увеличиваем). Гладкая функция
вместо ступенчатых правил «$\rho>0.75\Rightarrow\lambda/2$» устраняет
дёрганье $\lambda$ вблизи порогов.

Свойства: плавное уменьшение при хороших шагах, быстрый выход из плохой
области; у минимума на уровне шума $\lambda$ уходит вверх — это естественный
сигнал остановки, а не патология.


## 4. Адаптивное $\mu$: старт по кривизне + ужесточение по Пауэллу

**Автоматический старт.** «Мягко» или «жёстко» — понятие относительное:
сравнивать надо кривизну штрафа $\tfrac1\mu J_G^\top J_G$ с кривизной
измерительной части $H$. Требуя сопоставимости следов
($\operatorname{tr}(A) = \|A_{\text{факторы}}\|_F^2$), получаем автоматический
старт без единого ручного параметра:

$$
\mu_0 = \frac{\operatorname{tr}(J_G^\top J_G)}{\operatorname{tr}(J^\top J)}
= \frac{\|J_G\|_F^2}{\|J\|_F^2}\bigg|_{p_0} .
$$

На тестах: ЛВ $0.22$, Аттрактор $1.4$, реальные данные $1.1\cdot10^{-5}$ —
во всех случаях в рабочей зоне соответствующей задачи.

**Continuation (логика квадратичного штрафа Пауэлла).** После каждого
*принятого* шага $\mu$ ужесточается,
$\mu \leftarrow \max(\mu\cdot\texttt{mu\_dec},\ \mu_{\min})$, только когда
выполнены **оба** условия:

- $\|R_G\|^2 > \beta\,\|R_G^{\text{prev}}\|^2$ — невязка стыковки не падает
  сама, штрафу пора помогать ($\beta = $ `viol_target` $= 0.25$);
- $\|R\|^2 > \texttt{rss\_stall\_tol}\cdot\|R\|^2_{\text{prev}}$ —
  измерительная невязка застопорилась, глобализация закончилась
  (`rss_stall_tol` $= 0.99$).

Иначе $\mu$ не трогаем. При *отклонении* шага $\mu$ тоже не меняется —
длиной шага управляет $\lambda$ (в прежней схеме наоборот: на неудаче
меняли $\mu$, т.е. саму целевую).

**Зачем второй гейт.** Пока $\|R\|^2$ падает на порядки (первые итерации,
глобализация), стыковка естественно колеблется, и первое условие
срабатывает почти на каждом принятом шаге: $\mu$ утаптывается за 3–5
итераций, ограничения начинают доминировать, и решение запирается на
консистентной траектории вдали от измерений. Виднее всего на широких
шутах: Аттрактор при $N_{\text{shoot}}=5$ из $\theta_0=0$ **без** гейта не
сходится никогда (rel_err $1.18$ при идеальной стыковке
$\|R_G\|^2\sim10^{-10}$ — классическое «консистентно, но мимо измерений»),
**с** гейтом — rel_err $2.6\cdot10^{-4}$ и та же стыковка; при
$N_{\text{shoot}}=10/20$ гейт ничего не меняет. Эксперимент — §9,
закрепляющий тест — `test_attractor_converges_with_few_shoots`.

**Отвергнутая альтернатива** (проверена ниже): держать долю штрафа в
$\Phi_\mu$ постоянной, $\mu = \|R_G\|^2 / (\kappa\|R\|^2)$. На синтетике
работает при $\kappa\in[0.1,1]$, но отношение зависит от числа строк: на
реальных данных 16000 измерительных строк против 18 строк стыковки дают
$\mu_0\approx 10^{-6}$ — жёсткая стыковка с первого шага и вдвое худшая доля
принятых шагов. Оставлена в реализации как `mu_rule='ratio'` для сравнения.


## 5. Принятие шага и остановка

**Принятие:** $\rho > 0$ по $\Phi_\mu$ с тем же $\mu$, что использовался в
шаге. Никаких «$\le 1.1\cdot$best»: относительно текущей merit-функции
стоимость не растёт по построению. Поскольку $\mu$ меняется между
итерациями, $\Phi_\mu$ тоже меняется — монотонность гарантируется
поитерационно; для методов continuation это стандартная ситуация.

**Остановка** (все условия автоматические, `n_iter` становится верхней
границей, а не обязательным бюджетом):

- $\mathrm{pred} < \varepsilon\,\Phi_\mu$ — модель не предсказывает значимого
  уменьшения;
- серия подряд отклонённых шагов ($\lambda$-эскалация упёрлась в уровень
  шума);
- два подряд принятых шага с относительным уменьшением merit $<10^{-10}$
  (стагнация).


## 6. Алгоритм целиком

Реализация — `gauss_newton/adaptive.py` (`run_optimization_adaptive`),
тесты — `pytests/adaptive_test.py`. Ниже — полное описание одной итерации
со ссылками на параграфы, где выведена каждая формула.

**Вход:** `theta_full` $= p_0 = [\theta_0; c_1 \dots c_T]$ и объект `problem`,
умеющий вернуть нормальные уравнения в точке — либо через `solve` (тогда
$H = J^\top J$, $g = J^\top R$ считаются из построенной $J$), либо через
`normal_equations` (H и g копятся по измерениям, $J$ не строится). Параметры
по умолчанию: $\lambda_0 = 10^{-3}$, `mu_dec` $= 0.5$,
$\beta = $ `viol_target` $= 0.25$, `rss_stall_tol` $= 0.99$,
`max_rejects` $= 8$; `n_iter` — верхняя граница числа итераций.

**Инициализация.**

1. $(H, g, J_G, R_G) \leftarrow$ нормальные уравнения в $p_0$ — один полный
   расчёт невязок и чувствительностей.
2. $\mu \leftarrow \|J_G\|_F^2 / \operatorname{tr}(H)$ — старт по кривизне
   (§4; $\operatorname{tr}(H) = \|J\|_F^2$, сама $J$ не нужна);
   $\lambda \leftarrow \lambda_0$; $\nu_{\mathrm{esc}} \leftarrow 2$;
   $V \leftarrow \|R_G\|^2$, $S \leftarrow \|R\|^2$ (память для правила
   Пауэлла и его гейта).

**Итерация $k$:**

3. **Шаг.** Решить седловую систему (§1) с демпфером
   $D = \lambda_{\mathrm{reg}} I + \lambda\operatorname{diag}(H)$
   $\Rightarrow$ $\delta$; посчитать
   $\mathrm{pred} = \delta^\top(\tilde g + D\delta)$ (§3). Если $\delta$
   не конечен или $\mathrm{pred} \le 0$ (симптом сбоя факторизации) —
   считать итерацию отказом и перейти к шагу 7.
4. **Проба.** $p_{\mathrm{trial}} = p + \delta$; пересчитать нормальные
   уравнения в пробной точке — единственный дорогой вызов итерации
   (интегрирование чувствительностей); факторизация из шага 3 на его фоне
   пренебрежима. Если интегратор не справился (например, Ньютон коллокаций
   не сошёлся) — это отказ шага, а не авария.
5. **Gain ratio** (§3): $\rho = \big(\Phi_\mu(p) -
   \Phi_\mu(p_{\mathrm{trial}})\big) / \mathrm{pred}$, где
   $\Phi_\mu = \|R\|^2 + \tfrac1\mu\|R_G\|^2$ с **тем же** $\mu$, что в
   шаге 3 (§5). NaN/Inf в пробной точке $\Rightarrow \rho = -\infty$.
6. **Принятие** ($\rho > 0$): $p \leftarrow p_{\mathrm{trial}}$ вместе с
   $(H, g, J_G, R_G)$;
   $\lambda \leftarrow \lambda\cdot\max(\tfrac13,\, 1-(2\rho-1)^3)$,
   $\nu_{\mathrm{esc}} \leftarrow 2$ (Нильсен, §3);
   обновление $\mu$ (Пауэлл с гейтом, §4): если $\|R_G\|^2 > \beta V$
   **и** $\|R\|^2 > \texttt{rss\_stall\_tol}\cdot S$ —
   $\mu \leftarrow \max(\mu\cdot\texttt{mu\_dec},\ \mu_{\min})$;
   затем $V \leftarrow \|R_G\|^2$, $S \leftarrow \|R\|^2$.
7. **Отказ** ($\rho \le 0$): $p, H, g, \mu$ не меняются;
   $\lambda \leftarrow \lambda\cdot\nu_{\mathrm{esc}}$,
   $\nu_{\mathrm{esc}} \leftarrow 2\nu_{\mathrm{esc}}$. Отклонённый шаг
   тоже стоил один пересчёт, поэтому эскалация идёт удваивающимся
   $\nu_{\mathrm{esc}}$ — выход из плохой области за $O(\log)$ отказов.
8. **Остановка** (§5): `max_rejects` отказов подряд; два подряд принятых
   шага с относительным уменьшением merit $< 10^{-10}$ (стагнация);
   $\mathrm{pred} < 10^{-12}\,\Phi_\mu$.

**Роли параметров:**

| параметр | за что отвечает | как меняется | дефолт |
|---|---|---|---|
| $\lambda$ | длина шага (доверие к линеаризации) | Нильсен по $\rho$ (§3) | старт $10^{-3}$, дальше сам |
| $\mu$ | вес стыковки, continuation (§2) | Пауэлл по $\|R_G\|^2$ с гейтом по $\|R\|^2$ (§4) | старт по кривизне, дальше сам |
| $\nu_{\mathrm{esc}}$ | темп эскалации $\lambda$ при отказах | $\times 2$ за отказ, сброс при успехе | 2 |
| $\beta$ (`viol_target`) | порог «стыковка падает сама» | фиксирован | 0.25 |
| `rss_stall_tol` | гейт «измерения выжаты» для ужесточения $\mu$ (§4) | фиксирован | 0.99 |
| `mu_dec` | темп ужесточения $\mu$ | фиксирован | 0.5 |
| `rho_accept` | порог принятия шага | фиксирован | 0 |

Настраиваемых руками размерных величин не осталось: $\lambda$ и $\mu$
самокалибруются, а $\beta$, `rss_stall_tol`, `mu_dec`, `rho_accept`
безразмерны и слабочувствительны (эксперименты §7: рабочий диапазон
`mu_dec` $\in [0.1, 0.5]$ без изменения результата).

**Типичная траектория** (видно на графиках §7): (i) первые 1–3 итерации —
большие шаги при мягкой стыковке, глобализация ($\mu$ стоит на месте:
гейт по $\|R\|^2$ закрыт, пока измерительная невязка падает); (ii) середина —
$\mu$ систематически ужесточается («лесенка» на графике — срабатывания
правила Пауэлла), $\lambda$ плавно падает по мере роста доверия к
линеаризации; (iii) финал — merit на уровне шума данных, $\rho \to 0$,
$\lambda$ эскалирует, срабатывает остановка.


### Разбор `gn_step` (функция шага)

Функция делает два дела: решает седловую систему и возвращает
$\mathrm{pred}$ для gain ratio. На вход — объект `NormalEquations`
($H$, $g$, $J_G$, $R_G$; как они получены — через большую $J$ или
накоплением — шагу безразлично, см. врезку ниже). По строкам:

1. **Демпфер** `D = lambda_reg*I + lam*diags(maximum(H.diagonal(), 1e-10))`.
   Floor $10^{-10}$ на диагонали: если к какому-то параметру невязки локально
   нечувствительны (нулевой столбец $J$), его элемент $\operatorname{diag}(H)$
   равен нулю — без floor этот параметр получил бы нулевое демпфирование и
   произвольно большой шаг вдоль ненаблюдаемого направления.
2. **Седловая форма, а не исключённая.** Можно было бы собрать
   $\tilde H = H + \tfrac1\mu J_G^\top J_G$ и решать её напрямую, но при
   $\mu \to 0$ множитель $1/\mu$ раздувает число обусловленности как
   $O(1/\mu)$ (при $\mu = 10^{-8}$ — потеря $\sim$8 знаков). В седловой форме
   $\mu$ входит **линейно** в блок $-\mu I$, и разреженный LU (`spsolve`)
   устойчив во всём рабочем диапазоне. Это тот же приём, что в
   прежней схемы, — поэтому в `pytests/adaptive_test.py` есть тест
   `test_step_matches_dense_saddle_solve`: при одинаковых
   $(\mu, \lambda, \lambda_{\mathrm{reg}})$ решения обеих функций совпадают
   до допуска решателя.
3. **Градиент.** $\tilde g = g + \tfrac1\mu J_G^\top R_G = -\tfrac12
   \nabla\Phi_\mu$ (антиградиент), где $g = J^\top R$: при принятых в коде
   знаках невязок ($R = W(y-h)$, $R_G = -G$) шаг $p + \delta$ идёт в сторону
   убывания $\Phi_\mu$ — поэтому в правой части системы стоят $g$ и $R_G$
   с плюсом.
4. **pred** $= \delta^\top\tilde g + \delta^\top D\delta$ — вывод в §3;
   $\mathrm{pred} \le 0$ означает численный сбой решения и трактуется
   вызывающим кодом как отказ шага (тест
   `test_pred_positive_across_regimes` проверяет $\mathrm{pred} > 0$ по
   сетке $\mu \in [10^{-6}, 10^{2}]$, $\lambda \in [10^{-6}, 1]$).
5. **Возврат** $\delta$ целиком — приращение и по $\theta$, и по всем $c_j$;
   контракт тот же, что был у `compute_delta_gn`.

Стоимость: одна разреженная LU-факторизация матрицы размера
$(n_p + n_{\mathrm{cont}}) \times (n_p + n_{\mathrm{cont}})$ — пренебрежимо
мало по сравнению с расчётом $H$ и $g$ (интегрирование чувствительностей).

> **Два слоя.** Шаг и цикл (`gauss_newton/adaptive.py`) отделены от способа
> получить $H$ и $g$ (`gauss_newton/normal_equations.py`). Класс
> `NormalEquations` даёт их либо из готовой $J$ (`from_jacobian`: $H = J^\top J$),
> либо накоплением по измерениям без построения $J$ (`AccumulateMixin`,
> классы `MultipleShootingAccum` / `CollocationShootingAccum` — вывод в
> `collocation.ipynb`). Цикл в обоих случаях один и тот же:
> `run_optimization_adaptive(problem, theta_full)` сам выбирает путь по типу
> задачи.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from commom_utils.systems import LotkaVoltera, Attractor
from commom_utils.ode_system import SyntheticDataGenerator
from gauss_newton.problem import MultipleShooting
# слой оптимизации (шаг + цикл) и слой нормальных уравнений (H, g) — см. §6
from gauss_newton.adaptive import gn_step, run_optimization_adaptive
from gauss_newton.normal_equations import NormalEquations

CONFIGS = {
    'LotkaVolterra': dict(cls=LotkaVoltera,
                          true=np.array([1.2, 0.4, 0.3, 0.1]),
                          x0=np.array([6.0, 5.0]), interval=(0.0, 4.0),
                          n_meas=50, sigma=0.01, N_shoot=5,
                          theta_init=np.array([1.0, 0.5, 0.2, 0.05])),
    'Attractor': dict(cls=Attractor,
                      true=np.array([10.0, 28.0, 8.0 / 3.0]),
                      x0=np.array([1.0, 1.0, 1.0]), interval=(0.0, 5.0),
                      n_meas=100, sigma=0.01, N_shoot=20,
                      theta_init=np.array([0.0, 0.0, 0.0])),
}


def gen_data(cfg, seed=0):
    np.random.seed(seed)
    system = cfg['cls']()
    gen = SyntheticDataGenerator(system, sigma=cfg['sigma'],
                                 perturb_initial=True, perturbation_scale=0.0,
                                 use_jax=True)
    t_b, meas_b, _, _ = gen.generate(c0=cfg['x0'], theta=cfg['true'],
                                     time_intervals=[cfg['interval']],
                                     n_measurements=cfg['n_meas'])
    return system, t_b[0], meas_b[0]


def fresh_problem(cfg, t, meas):
    prob = MultipleShooting(system=cfg['cls'](), N_shoot=cfg['N_shoot'],
                            gamma=np.ones(cfg['cls']().n_obs), c0_cost=1.0,
                            use_jax=True)
    prob.add_batch(meas, t)
    return prob


In [2]:
# Функция шага, разобранная выше (источник - gauss_newton/adaptive.py)
import inspect
print(inspect.getsource(gn_step))


def gn_step(ne, mu, lam, lambda_reg=0.0, lam_dual=None):
    """Шаг из mu-регуляризованной седловой системы + pred для gain ratio.

    Решается седловая система (mu-регуляризованная ККТ), плюс pred:
        [[H + D, J_G^T], [J_G, -mu I]] [delta; nu] = [g; R_G],
    D = lambda_reg*I + lam*diag(H). Седловая форма вместо исключённой
    (H + (1/mu) J_G^T J_G): при mu -> 0 исключённая теряет обусловленность
    как O(1/mu), а здесь mu входит линейно.

    pred = delta^T (g_eff + D delta) >= 0 — предсказанное моделью уменьшение
    Phi_mu (вывод в adaptive_regularization.ipynb, §3); pred <= 0 возможен
    только при численном сбое и трактуется циклом как отказ шага.

    lam_dual — оценка множителей Лагранжа для augmented-Lagrangian-режима
    (эксперименты: adaptive_regularization.ipynb, раздел про AL). При
    заданном lam_dual RHS блока ограничений сдвигается на -mu*lam_dual —
    это шаг ГН для L_A = ||R||^2 - 2 lam_dual^T R_G + (1/mu)||R_G||^2
    (у нас R_G = -G), в g_eff добавляется

## 7. Эксперименты (Лотка–Вольтерра, Аттрактор Лоренца)

Сравнение на синтетике: baseline (`run_optimization_manual`, воспроизведён ниже) с перебором $\mu_0$ по
шести порядкам против адаптивной схемы с перебором её безразмерных
гиперпараметров (`mu_dec` для правила по кривизне, $\kappa$ для отвергнутого
правила по невязке). Настройка: шум $\sigma = 0.01$, у Аттрактора старт из
$\theta_0 = 0$ (заведомо плохое приближение), лимит 40 итераций. В таблице:
`iters` — фактически выполнено итераций, `accepted` — из них принято шагов,
`to_5pct` — итерация достижения 5% относительной ошибки $\theta$,
`mu` — траектория $\mu$ от старта до финиша.


In [3]:
def rel_err(theta, true):
    return float(np.max(np.abs((theta[:len(true)] - true) / true)))


def first_below(errs, tol=0.05):
    for i, e in enumerate(errs):
        if e < tol:
            return i
    return None


def run_optimization_manual(problem, theta_full, mu0, n_iter=40, lambda_=1e-3,
                            lambda_reg=0.0, mu_dec=0.7, mu_min=1e-6):
    """Прежняя схема с РУЧНЫМ расписанием mu — предмет сравнения этого ноутбука.

    До рефакторинга жила в библиотеке (run_optimization + compute_delta_gn).
    Оттуда удалена: в библиотеке остались один шаг (gn_step) и один цикл
    (run_optimization_adaptive), а вторая копия той же математики разъезжалась
    с первой. Здесь схема воспроизведена локально — ровно потому, что
    сравнение с ней и есть содержание ноутбука. Сам ШАГ берётся из библиотеки,
    так что сравниваются именно стратегии mu/lambda, а не две реализации ГН.

    Отличия от адаптивной схемы (см. §1):
      - mu умножается на mu_dec после КАЖДОГО принятого шага, независимо от
        того, падает ли невязка стыковки сама;
      - шаг принимается по грубому cost <= 1.1 * best, а не по rho > 0 для той
        Phi_mu, для которой шаг посчитан (то есть принимается и УХУДШЕНИЕ);
      - lambda фиксировано;
      - остановки по существу нет — всегда n_iter итераций либо 3 отказа подряд.

    Возвращает историю theta по итерациям.
    """
    theta_full = theta_full.copy()
    mu, fails = mu0, 0
    ne = NormalEquations.from_jacobian(*problem.solve(theta_full))
    best = ne.cost()
    hist = [theta_full.copy()]
    for _ in range(n_iter):
        delta, _ = gn_step(ne, mu, lambda_, lambda_reg)
        trial = theta_full + delta
        if not np.all(np.isfinite(trial)):
            break
        ne_trial = NormalEquations.from_jacobian(*problem.solve(trial))
        cost = ne_trial.cost()
        if np.isfinite(cost) and cost <= best * 1.1:
            theta_full, ne, best, fails = trial, ne_trial, cost, 0
            mu = max(mu * mu_dec, mu_min)          # безусловное ужесточение
        else:
            fails += 1
            mu = max(mu / mu_dec, mu_min)
            if fails >= 3:
                break
        hist.append(theta_full.copy())
    return hist


RUNS = {}
rows = []
for name, cfg in CONFIGS.items():
    system, t, meas = gen_data(cfg)
    for mu0 in [1e-2, 1e0, 1e2, 1e4]:
        prob = fresh_problem(cfg, t, meas)
        th0 = prob.make_full_theta(cfg['theta_init'])
        errs = [rel_err(th, cfg['true'])
                for th in run_optimization_manual(prob, th0, mu0=mu0)]
        label = f'baseline mu0={mu0:g}'
        RUNS[(name, label)] = dict(errs=errs)
        rows.append(dict(system=name, method=label, iters=len(errs) - 1,
                         accepted='-', to_5pct=first_below(errs),
                         final_err=errs[-1], mu='-'))
    for label, kw in [('adaptive ratio k=0.1', dict(mu_rule='ratio', kappa=0.1)),
                      ('adaptive ratio k=1', dict(mu_rule='ratio', kappa=1.0)),
                      ('adaptive curv dec=0.1', dict(mu_dec=0.1)),
                      ('adaptive curv dec=0.2', dict(mu_dec=0.2)),
                      ('adaptive curv dec=0.5', dict(mu_dec=0.5))]:
        prob = fresh_problem(cfg, t, meas)
        th0 = prob.make_full_theta(cfg['theta_init'])
        th, hist = run_optimization_adaptive(prob, th0, **kw)
        errs = [rel_err(x, cfg['true']) for x in hist['theta']]
        RUNS[(name, label)] = dict(errs=errs, hist=hist)
        rows.append(dict(system=name, method=label, iters=len(errs) - 1,
                         accepted=len(hist['accepted']),
                         to_5pct=first_below(errs), final_err=errs[-1],
                         mu=f"{hist['mu'][0]:.1e} -> {hist['mu'][-1]:.1e}"))

table = pd.DataFrame(rows)
table['final_err'] = table['final_err'].map('{:.2e}'.format)
table


,system,method,iters,accepted,to_5pct,final_err,mu
0,LotkaVolterra,baseline mu0=0.01,40,-,2.0,9.54e-03,-
1,LotkaVolterra,baseline mu0=1,40,-,2.0,9.54e-03,-
2,LotkaVolterra,baseline mu0=100,40,-,3.0,9.54e-03,-
3,LotkaVolterra,baseline mu0=10000,40,-,3.0,9.41e-03,-
4,LotkaVolterra,adaptive ratio k=0.1,27,11,2.0,9.54e-03,1.8e+00 -> 1.0e-08
5,LotkaVolterra,adaptive ratio k=1,30,8,2.0,9.54e-03,1.8e-01 -> 1.0e-08
6,LotkaVolterra,adaptive curv dec=0.1,40,18,2.0,9.54e-03,2.2e-01 -> 1.0e-08
7,LotkaVolterra,adaptive curv dec=0.2,40,17,2.0,9.54e-03,2.2e-01 -> 1.1e-07
8,LotkaVolterra,adaptive curv dec=0.5,30,16,2.0,9.54e-03,2.2e-01 -> 5.4e-05
9,Attractor,baseline mu0=0.01,7,-,NaN,9.59e-01,-


Чтение таблицы:

- **baseline, Аттрактор, $\mu_0=0.01$** — расходимость (NaN на 7-й итерации,
  `to_5pct=None`): цена неудачного ручного $\mu_0$. Остальные baseline
  сходятся, но всегда тратят все 40 итераций.
- **adaptive curv** — сходится при всех `mu_dec` из диапазона $[0.1, 0.5]$ на
  обеих системах, стартовое $\mu_0$ вычислено автоматически, останавливается
  сам.
- **adaptive ratio** — на синтетике тоже работает ($\kappa\in[0.1,1]$), но
  см. §8: на реальных данных его стартовое $\mu$ получается на 5 порядков
  жёстче нужного.


In [4]:
C_BLUE, C_ORANGE, C_AQUA = '#2a78d6', '#eb6834', '#1baf7a'

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
for label, color in [('baseline mu0=0.01', C_ORANGE),
                     ('baseline mu0=100', C_BLUE),
                     ('adaptive curv dec=0.5', C_AQUA)]:
    errs = RUNS[('Attractor', label)]['errs']
    ax.semilogy(range(len(errs)), errs, color=color, lw=2, label=label)
ax.axhline(0.05, color='#52514e', lw=1, ls=':')
ax.set_xlabel('итерация'); ax.set_ylabel('max относительная ошибка θ')
ax.set_title('Аттрактор: сходимость')
ax.legend(frameon=False); ax.grid(alpha=0.25)

hist = RUNS[('Attractor', 'adaptive curv dec=0.5')]['hist']
ax = axes[1]
ax.semilogy(hist['mu'], color=C_AQUA, lw=2)
ax.set_xlabel('итерация'); ax.set_ylabel('mu')
ax.set_title('adaptive: траектория mu\n(старт по кривизне, ужесточение по Пауэллу)')
ax.grid(alpha=0.25)

ax = axes[2]
ax.semilogy(hist['lam'], color=C_AQUA, lw=2)
ax.set_xlabel('итерация'); ax.set_ylabel('lambda')
ax.set_title('adaptive: траектория lambda (Нильсен)\nрост в конце = уровень шума достигнут')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


<Figure size 1500x420 with 3 Axes>

## 8. Реальные данные (CeedEveron, латеральная динамика)

Конфигурация — из `experiments/lateral_movement_ceed_dynamic.ipynb`
(8000 точек, `DynamicModelRearAxle`, $N_{\text{shoot}}=10$, оценивается
$\theta = [C_f, C_r, a_{\mathrm{rel}}, I_z, GR]$; в ноутбуке вручную
подобраны $\mu_0=0.03$, `mu_dec=0.5`, $\lambda=0.05$,
$\lambda_{\mathrm{reg}}=10^{-4}$, `n_iter=30`). Запуск — отдельным скриптом
с `MultipleShooting(use_jax=True)`; результаты:

| метод | итераций | принято | время | стоимость | $\theta$ |
|---|---|---|---|---|---|
| baseline $\mu_0=0.03$ (ноутбук) | 30 | — | 20.8 c | 3.3206e4 | [2.436, 2.824, 0.415, 0.050, 12.481] |
| baseline $\mu_0=100$ | 30 | — | 20.5 c | 3.3204e4 | [2.425, 2.788, 0.414, 0.050, 12.481] |
| baseline $\mu_0=10^{-4}$ | 30 | — | 21.0 c | 3.3207e4 | [2.443, 2.827, 0.414, 0.050, 12.481] |
| adaptive ratio $\kappa=0.1$ | 24 | 8 | 17.8 c | 3.3215e4 | [2.331, 3.482, 0.469, 0.056, 12.482] |
| adaptive curv `mu_dec=0.2` | 30 | 13 | 20.9 c | 3.3208e4 | [2.230, 2.664, 0.431, 0.048, 12.477] |
| adaptive curv `mu_dec=0.5` | 24 | 11 | 17.5 c | 3.3207e4 | [2.240, 2.660, 0.429, 0.048, 12.477] |

Наблюдения:

- На этих данных baseline нечувствителен к $\mu_0$ (от $10^{-4}$ до $100$
  одинаково) — ручная настройка здесь не была нужна, но узнать это можно
  было только перебором.
- `adaptive curv` выходит на ту же стоимость (3.3207e4 против 3.3204e4,
  разница $10^{-4}$ отн.) без единого ручного параметра и с самоостановкой
  (24 итерации вместо обязательных 30).
- Разные $\theta$ при равной стоимости — плоская долина: $C_f$ и $C_r$ слабо
  идентифицируемы порознь (согласуется с широкими доверительными
  интервалами); это свойство задачи, а не метода.
- `adaptive ratio` стартует с $\mu\approx 10^{-6}$ (16000 измерительных строк
  против 18 строк стыковки перекашивают отношение норм невязок) — доля
  принятых шагов падает до 8/24 и стоимость чуть хуже. Поэтому правило
  по кривизне, а не по невязке.


## 9. Augmented Lagrangian: множители вместо $\mu \to 0$

Точная задача — МНК с ограничениями непрерывности:

$$
\min_p \ \|R(p)\|^2 \quad \text{s.t.} \quad G(p) = 0,
$$

где $G$ — невязки стыковки (в коде $R_G = -G$, см. §1). Модифицированная
функция Лагранжа (в конвенции ноутбука — без $\tfrac12$ перед квадратами,
отсюда множитель 2 при линейном члене):

$$
L_A(p, \lambda;\, \mu) = \|R\|^2 + 2\lambda^\top G + \tfrac1\mu\|G\|^2
= \|R\|^2 - 2\lambda^\top R_G + \tfrac1\mu\|R_G\|^2 .
$$

При $\lambda = 0$ это $\Phi_\mu$ из §2: текущая схема — частный случай AL с
нулевыми множителями. Классический результат (Хестенс 1969, Пауэлл 1969;
Бертсекас 1982): существует порог $\bar\mu > 0$ такой, что при любом
**фиксированном** $\mu < \bar\mu$ минимум $L_A(\cdot, \lambda^*)$ — это
точное решение задачи с ограничениями. Поэтому у AL два эквивалентных пути
к нерегуляризированной задаче: у штрафа — $\mu \to 0$ (continuation, §4),
у AL — итерации множителей $\lambda_{k+1} = \lambda_k + G(p_{k+1})/\mu$
(первопорядковое обновление Хестенса–Пауэлла) при умеренном $\mu$.

**Шаг ГН для $L_A$.** Линеаризуем $R(p+\delta) \approx R - J\delta$ и
$G(p+\delta) \approx -R_G + J_G\delta$, подставим в $L_A$ и приравняем
градиент по $\delta$ нулю:

$$
\Big(H + D + \tfrac1\mu J_G^\top J_G\Big)\delta
= \underbrace{g - J_G^\top\lambda + \tfrac1\mu J_G^\top R_G}_{\tilde g_\lambda},
\qquad g = J^\top R .
$$

В седловой форме меняется **только правая часть блока ограничений**:

$$
\begin{bmatrix} H + D & J_G^\top \\ J_G & -\mu I \end{bmatrix}
\begin{bmatrix} \delta \\ \nu \end{bmatrix}
= \begin{bmatrix} g \\ R_G - \mu\lambda \end{bmatrix} .
$$

Двойственное решение $\nu = \lambda + (J_G\delta - R_G)/\mu$ — это
первопорядковое обновление множителей, вычисленное на линеаризованном
ограничении: седловая система *бесплатно* отдаёт готовую $\lambda_{k+1}$.
Формула $\mathrm{pred}$ из §3 не меняется —
$\mathrm{pred} = \delta^\top(\tilde g_\lambda + D\delta)$. Всё это
реализовано в библиотеке: `gn_step(..., lam_dual=lam)` возвращает
$(\delta, \mathrm{pred}, \nu)$; при $\lambda = 0$ шаг побитово совпадает с
обычным (тест `test_step_with_multipliers_matches_dense_saddle_solve` —
сверка с плотным решением той же системы).

**Эксперименты.** Полигон — случай, где penalty-схеме труднее всего и где
пользователь наблюдал несходимость: Аттрактор с $N_{\text{shoot}} = 5$
(широкие шуты $\Rightarrow$ сильно нелинейное $G$), $\theta_0 = 0$. Пять
стратегий на одном и том же `gn_step`; отличаются только политикой
$(\mu, \lambda)$-обновлений.


In [5]:
from gauss_newton.normal_equations import normal_equations_of


def run_optimization_al(problem, theta_full, n_iter=60, lam0=1e-3,
                        update='always', eta0_frac=0.1, eta_dec=0.3,
                        lam_max=1e6):
    """AL-цикл ЛОКАЛЬНО в ноутбуке (в библиотеке его нет — итог см. ниже).

    Каркас тот же, что у run_optimization_adaptive (Нильсен-lambda, принятие
    по gain ratio), отличия ровно те, что делают из штрафа AL:
      - mu ФИКСИРОВАН на старте по кривизне — ужесточения нет вовсе,
        точную стыковку должны обеспечить множители;
      - merit — L_A = ||R||^2 - 2 lam^T R_G + (1/mu)||R_G||^2 с ТЕКУЩИМИ
        множителями (сравнение до/после шага честное: lam обновляется только
        после принятия);
      - шаг — gn_step(..., lam_dual=lam_d): сдвиг правой части R_G - mu*lam,
        возврат двойственного решения nu;
      - обновление множителей после принятого шага, по правилу `update`:
          'always' — lam <- nu после каждого принятого шага;
          'eta'    — LANCELOT-гейт: первопорядковое lam <- lam - R_G/mu
                     только когда ||R_G|| <= eta; после обновления
                     eta <- eta*eta_dec, иначе lam не трогается;
          'never'  — lam = 0 навсегда: чистый штраф с замороженным mu
                     (базовая линия для сравнения).
    Возвращает (theta_full, ne, mu, lam_d, n_accepted).
    """
    theta_full = theta_full.copy()
    ne = normal_equations_of(problem, theta_full)
    mu = float(np.clip(ne.mu_curvature(), 1e-8, 1e8))
    lam_d = np.zeros(ne.n_cont)
    lam, nu_esc, n_acc = lam0, 2.0, 0
    eta = eta0_frac * np.sqrt(ne.cont_sq())

    def merit(ne_):
        return ne_.rss - 2.0 * float(lam_d @ ne_.R_G) + ne_.cont_sq() / mu

    for _ in range(n_iter):
        delta, pred, nu = gn_step(ne, mu, lam, lam_dual=lam_d)
        if not (np.all(np.isfinite(delta)) and pred > 0):
            lam = min(lam * nu_esc, 1e10); nu_esc *= 2.0
            continue
        try:
            ne_trial = normal_equations_of(problem, theta_full + delta)
            rho = (merit(ne) - merit(ne_trial)) / pred
        except RuntimeError:
            rho = -np.inf
        if np.isfinite(rho) and rho > 0:
            theta_full, ne = theta_full + delta, ne_trial
            lam = max(lam * max(1 / 3, 1 - (2 * rho - 1) ** 3), 1e-12)
            nu_esc, n_acc = 2.0, n_acc + 1
            if update == 'always':
                lam_d = np.clip(nu, -lam_max, lam_max)
            elif update == 'eta' and np.sqrt(ne.cont_sq()) <= eta:
                lam_d = np.clip(lam_d - ne.R_G / mu, -lam_max, lam_max)
                eta *= eta_dec
        else:
            lam = min(lam * nu_esc, 1e10); nu_esc *= 2.0
    return theta_full, ne, mu, lam_d, n_acc


In [6]:
# Полигон — худший для penalty-схемы случай: широкие шуты (N_shoot=5),
# сильно нелинейное G, старт из theta = 0
cfg_hard = dict(CONFIGS['Attractor'], N_shoot=5)
system, t, meas = gen_data(cfg_hard)

al_rows = []


def report(label, theta, ne, mu, lam_d, n_acc):
    al_rows.append(dict(
        strategy=label, rel_err=rel_err(theta, cfg_hard['true']),
        rss=ne.rss, cont_sq=ne.cont_sq(), mu_end=mu,
        lam_norm=float(np.linalg.norm(lam_d)) if lam_d is not None else 0.0,
        accepted=n_acc))


# 1-2: библиотечная penalty-схема — без гейта rss (прежнее поведение) и с ним
for label, kw in [('penalty schedule, БЕЗ гейта rss', dict(rss_stall_tol=0.0)),
                  ('penalty schedule + гейт rss (библиотека)', dict())]:
    prob = fresh_problem(cfg_hard, t, meas)
    th0 = prob.make_full_theta(cfg_hard['theta_init'])
    th, hist = run_optimization_adaptive(prob, th0, n_iter=80,
                                         track_covariance=False, **kw)
    ne_fin = NormalEquations.from_jacobian(*prob.solve(th))
    report(label, th, ne_fin, hist['mu'][-1], None, len(hist['accepted']))

# 3-5: AL-варианты и штраф с замороженным mu (все — на одном gn_step)
for label, update in [('AL: lam <- nu каждый принятый шаг', 'always'),
                      ('AL: eta-гейт (LANCELOT)', 'eta'),
                      ('penalty, mu заморожен (lam = 0)', 'never')]:
    prob = fresh_problem(cfg_hard, t, meas)
    th0 = prob.make_full_theta(cfg_hard['theta_init'])
    th, ne_fin, mu_fin, lam_d, n_acc = run_optimization_al(prob, th0,
                                                           update=update)
    report(label, th, ne_fin, mu_fin, lam_d, n_acc)

al_table = pd.DataFrame(al_rows)
for col in ('rel_err', 'rss', 'cont_sq', 'mu_end', 'lam_norm'):
    al_table[col] = al_table[col].map('{:.2e}'.format)
al_table


,strategy,rel_err,rss,cont_sq,mu_end,lam_norm,accepted
0,"penalty schedule, БЕЗ гейта rss",1.18e+00,9.05e+03,8.22e-09,1.00e-08,0.00e+00,54
1,penalty schedule + гейт rss (библиотека),2.65e-04,2.95e-02,4.32e-10,5.92e-08,0.00e+00,41
2,AL: lam <- nu каждый принятый шаг,1.47e+00,6.62e+03,1.74e+00,1.24e-01,6.52e+01,39
3,AL: eta-гейт (LANCELOT),3.66e-02,3.88e+01,8.16e+00,1.24e-01,3.91e+01,18
4,"penalty, mu заморожен (lam = 0)",2.86e-04,2.86e-02,2.27e-05,1.24e-01,0.00e+00,18


Чтение таблицы (числа — из ячейки выше; сид фиксирован):

- **penalty без гейта rss** — то самое «консистентно, но мимо измерений»:
  стыковка затянута ($\|R_G\|^2\sim10^{-8}$), но $\|R\|^2\sim 9\cdot10^3$ и
  rel_err $\approx 1.2$ — решение заперто на чужой траектории, и добавка
  итераций не помогает. Это прежнее поведение библиотеки
  (`rss_stall_tol=0`).
- **penalty с гейтом** (текущая библиотечная схема) — единственная
  стратегия, дающая и измерения (rel_err $\sim 3\cdot10^{-4}$), и машинную
  стыковку ($\|R_G\|^2\sim 4\cdot10^{-10}$).
- **AL, $\lambda \leftarrow \nu$ с первой итерации** — расходится: вдали от
  оптимума двойственное решение $\nu$ огромно и вносит в $L_A$ большой
  линейный член с произвольным знаком — задача портится раньше, чем
  глобализуется.
- **AL с $\eta$-гейтом (LANCELOT)** — лучше, но застревает на полпути:
  первое же обновление $\lambda$ происходит при ещё большой невязке и
  уводит в сторону.
- **штраф с замороженным $\mu$** ($\lambda \equiv 0$) — сходится к
  измерениям, но стыковка остаётся на уровне $\|R_G\|^2 \sim 10^{-5} =
  O(\mu\|\nu^*\|)$ — ровно то, что AL должен был бы дочистить множителями…
  но не дочищает (см. ниже).

**Почему множители здесь не работают.** ГН-приближение гессиана $L_A$ не
содержит членов со вторыми производными ограничений
$\sum_i (\lambda_i + G_i/\mu)\,\nabla^2 G_i$. У Лоренца на широких шутах
$\nabla^2 G$ (вторая производная потока по начальному состоянию) огромна:
вблизи решения фактическая кривизна мерита $\sim 2\cdot10^4$ при
$\mathrm{pred}\sim 10^{-8}$ — модель предсказывает крошечное уменьшение,
реальный $L_A$ на этом шаге растёт, $\rho < 0$, и все $\lambda$-шаги
отклоняются: феасибилити стоит на месте при любом правиле обновления.
Continuation по $\mu$ этой проблемы не имеет: он не просит модель
предсказать эффект множителей, а просто перевзвешивает две группы невязок —
шаг остаётся «честным» ГН для $\Phi_\mu$, а седловая форма (§2) позволяет
гнать $\mu$ хоть до $10^{-8}$ без потери обусловленности.

**Вывод.** Для этой задачи с ГН-гессианом правильная реализация «релаксации
множителей» — это НЕ augmented Lagrangian, а квадратичный штраф в седловой
форме с continuation по $\mu$ и гейтом по $\|R\|^2$ (реализовано в
`run_optimization_adaptive`). AL стал бы конкурентоспособен только вместе с
точным гессианом ограничений (полноценный SQP со вторыми производными
потока) — это другой метод с другой ценой итерации. Инструменты для
дальнейших экспериментов оставлены: `gn_step(..., lam_dual)` в библиотеке и
`run_optimization_al` в этом ноутбуке.


## 10. Выводы

**Рекомендуемая схема** (всё проверено выше):

1. $\lambda$ — по gain ratio Нильсена; $\lambda_0 = 10^{-3}$, дальше сама.
2. $\mu$ — старт $\|J_G\|_F^2 / \operatorname{tr}(H)$, ужесточение по Пауэллу
   с гейтом по измерениям (`mu_dec=0.5`, `viol_target=0.25`,
   `rss_stall_tol=0.99`); при отклонении шага $\mu$ не трогать.
3. Принятие — $\rho > 0$ по $\Phi_\mu$ с тем же $\mu$, что в шаге.
4. Остановка — pred/стагнация/серия отказов; `n_iter` — только верхняя
   граница.
5. Множители Лагранжа НЕ итерируются: с ГН-гессианом они не продвигают
   феасибилити (§9), а continuation по $\mu$ в седловой форме продвигает
   без потери обусловленности.

**Что это даёт.**

- Убирает ручной подбор $\mu_0$ / `mu_dec` (и защищает от расходимости,
  которую даёт неудачный $\mu_0$ — Аттрактор, $\mu_0=0.01$).
- Не хуже baseline по скорости сходимости (до 5% за те же 2–4 итерации) и
  по финальному качеству на всех трёх задачах, включая реальные данные.
- Гейт `rss_stall_tol` расширяет область сходимости по числу шутов:
  Аттрактор при $N_{\text{shoot}}=5$ без гейта запирается на консистентной
  траектории вдали от измерений, с гейтом сходится (§4, §9).
- Самоостановка: меньше пересчётов на задачах, где сходимость быстрая.
- Гиперпараметры остались, но стали безразмерными и слабочувствительными:
  на всех трёх задачах работают одни и те же значения по умолчанию.

**Реализация — два слоя** (прежний цикл из библиотеки удалён, см. §1):

- `gauss_newton/normal_equations.py` — как получить $H$ и $g$:
  `NormalEquations.from_jacobian` (из построенной $J$) или `AccumulateMixin`
  (накопление по измерениям, $J$ не строится; классы
  `MultipleShootingAccum`, `CollocationShootingAccum`). Там же —
  `covariance_theta` (ковариация $\theta$ прямо из $H$).
- `gauss_newton/adaptive.py` — что с ними делать: `gn_step` (шаг; с
  `lam_dual` — AL-вариант шага, §9) и `run_optimization_adaptive` (цикл).
  Один цикл на оба пути.

**Тесты** (`python -m pytest pytests/`):

- `adaptive_test.py::test_step_matches_dense_saddle_solve` — при одинаковых
  $(\mu, \lambda, \lambda_{\mathrm{reg}})$ шаг совпадает с
  плотным `numpy.linalg.solve` (та же седловая система);
- `test_step_with_multipliers_matches_dense_saddle_solve` — то же для
  AL-шага с `lam_dual` (§9): сдвинутая правая часть, возврат $\nu$,
  побитовое совпадение с обычным шагом при $\lambda = 0$;
- `test_pred_positive_across_regimes` — $\mathrm{pred} > 0$ по сетке
  $\mu \in [10^{-6}, 10^{2}]$, $\lambda \in [10^{-6}, 1]$;
- `test_identification_adaptive[LotkaVolterra|Attractor]` — сквозная
  идентификация без ручного $\mu$; Аттрактор — из $\theta_0 = 0$, случай,
  где baseline с $\mu_0 = 0.01$ расходится;
- `test_attractor_converges_with_few_shoots` — Аттрактор,
  $N_{\text{shoot}}=5$: закрепляет гейт `rss_stall_tol` (§4);
- `test_beats_cost_at_true_parameters` — стоимость в найденной точке не выше
  стоимости в истинных параметрах (внешний эталон минимизации);
- `test_single_shooting_early_stop` — $J_G$ пусто (чистый LM), ранняя
  остановка на точных данных;
- `accumulated_test.py::test_agrees_with_dense_path` — путь через $J$ и путь
  с накоплением дают один результат ($<10^{-6}$);
- `collocation_accum_test.py` — идентификация на `CollocationShootingAccum`
  с визуализацией через `plot_solution`.

Пример использования:

```python
from gauss_newton.normal_equations import CollocationShootingAccum
from gauss_newton.adaptive import run_optimization_adaptive

problem = CollocationShootingAccum(system, N_shoot=10, gamma=gamma)
problem.add_batch(measured, t)
theta_opt, hist = run_optimization_adaptive(
    problem, problem.make_full_theta(theta0), n_iter=30)
# hist: theta / cost / mu / lam / r_meas / r_cont / ci_low / ci_high
#       по итерациям (готово для plot_solution), плюс accepted и n_solves
```

**Возможные развития:** augmented Lagrangian исследован в §9 — с ГН-гессианом
не выигрывает; стал бы осмысленным вместе со вторыми производными потока
(полноценный SQP — другая цена итерации). Открытым остаётся
$\lambda_{\mathrm{reg}}$ по принципу невязки/GCV для плохо обусловленных
задач (на текущих задачах не потребовалось).
